In [ ]:
import csv
# Data Structure: Tuple format (name, gold, silver, bronze, weighted_score)
def load_data(filename):
    """Load data as tuples"""
    with open(filename) as f:
        return [
            (row['Country'],
             int(row['Gold']),
             int(row['Silver']),
             int(row['Bronze']),
             3*int(row['Gold']) + 2*int(row['Silver']) + int(row['Bronze']))
            for row in csv.DictReader(f)
        ]

In [ ]:
# Linear Search Function
def linear_search(countries, name):
    """Search for a country by name (case-insensitive)."""
    for country in countries:
        if country[0].lower() == name.lower():
            return country
    return None  # If not found

In [ ]:
# Custom TimSort Implementation
MIN_MERGE = 32

def calc_min_run(n):
    """Calculate the minimum run length for TimSort."""
    r = 0
    while n >= MIN_MERGE:
        r |= n & 1
        n >>= 1
    return n + r

def insertion_sort(A, left=0, right=None):
    """Insertion sort following your pseudocode exactly."""
    if right is None:
        right = len(A) - 1

    for i in range(left + 1, right + 1):
        temp = A[i]
        hole = i - 1
        while hole >= left and A[hole][4] < temp[4]:  # Changed to descending order
            A[hole + 1] = A[hole]
            hole -= 1
        A[hole + 1] = temp

def merge_sort(A):
    """Merge sort following your pseudocode exactly."""
    n = len(A)
    if n < 2:
        return

    mid = n // 2
    left = A[:mid]
    right = A[mid:]

    merge_sort(left)
    merge_sort(right)
    merge(left, right, A)

def merge(L, R, A):
    """Merge function following your pseudocode exactly."""
    nL = len(L)
    nR = len(R)
    i = j = k = 0

    while i < nL and j < nR:
        if L[i][4] >= R[j][4]:  # Changed to descending order
            A[k] = L[i]
            i += 1
        else:
            A[k] = R[j]
            j += 1
        k += 1

    while i < nL:
        A[k] = L[i]
        i += 1
        k += 1

    while j < nR:
        A[k] = R[j]
        j += 1
        k += 1

def tim_sort(arr):
    """Sort countries by weighted score (descending) using TimSort."""
    n = len(arr)
    if n < MIN_MERGE:
        insertion_sort(arr)
        return arr

    min_run = calc_min_run(n)

    # Sort individual runs with insertion sort
    for start in range(0, n, min_run):
        end = min(start + min_run - 1, n - 1)
        insertion_sort(arr, start, end)

    # Merge runs using merge sort
    size = min_run
    while size < n:
        for left in range(0, n, 2 * size):
            mid = min(n - 1, left + size - 1)
            right = min(left + 2 * size - 1, n - 1)

            if mid < right:
                # Extract the runs to merge
                left_run = arr[left:mid + 1]
                right_run = arr[mid + 1:right + 1]
                merged = left_run + right_run
                merge_sort(merged)
                # Put merged back into array
                arr[left:right + 1] = merged
        size *= 2

    return arr

# Core Logic
def get_top_k(countries, k):
    """Get top K countries using TimSort."""
    if k <= 0:
        return []
    # First sort by weighted score
    sorted_by_score = tim_sort(countries.copy())

    return sorted_by_score[:k]

# User Interaction
def display_top(countries):
    print("\nRank | Country            | Gold | Silver | Bronze | Weighted")
    print("-----|--------------------|------|--------|--------|---------")
    for idx, (name, g, s, b, w) in enumerate(countries, 1):
        print(f"{idx:4} | {name[:18]:18} | {g:4} | {s:6} | {b:6} | {w:8}")

def compare_countries(data, name1, name2):
    """Compare two countries by weighted score using linear search."""
    country1 = linear_search(data, name1)
    country2 = linear_search(data, name2)

    if not country1 or not country2:
        print("Error: One or both countries not found!")
        return

    print(f"\nComparison:")
    print(f"{country1[0]}: {country1[4]} pts (G:{country1[1]}, S:{country1[2]}, B:{country1[3]})")
    print(f"{country2[0]}: {country2[4]} pts (G:{country2[1]}, S:{country2[2]}, B:{country2[3]})")

    if country1[4] > country2[4]:
        print(f"{country1[0]} ranks higher!")
    elif country1[4] < country2[4]:
        print(f"{country2[0]} ranks higher!")
    else:
        print("Both countries have equal scores!")

# Main Program (unchanged)
def main():
    global countries_sorted_by_name
    data = load_data('olympics2024.csv')
    countries_sorted_by_name = sorted(data.copy(), key=lambda x: x[0].lower(), reverse=True)

    while True:
        print("\n==== Olympics 2024 Ranking Based on Weighted Score ====")
        print("1. View Top K Countries")
        print("2. Compare Two Countries")
        print("3. Exit")

        choice = input("Choose an option (1-3): ")

        if choice == '1':
            try:
                k = int(input("Enter the number of top countries you want to find: "))
                if 0 < k <= len(data):
                    top_k = get_top_k(data, k)
                    display_top(top_k)
                elif k <= 0:
                    print("Please enter a positive number!")
                else:
                    print("Invalid number of countries!")
            except ValueError:
                print("Please enter a valid number!")

        elif choice == '2':
            name1 = input("Enter first country: ").strip()
            name2 = input("Enter second country: ").strip()
            compare_countries(data, name1, name2)

        elif choice == '3':
            print("Feel free to visit again!")
            break

        else:
            print("Invalid choice!")

if __name__ == "__main__":
    main()


==== Olympics 2024 Ranking Based on Weighted Score ====
1. View Top K Countries
2. Compare Two Countries
3. Exit
Choose an option (1-3): 1
Enter the number of top countries you want to find: 32

Rank | Country            | Gold | Silver | Bronze | Weighted
-----|--------------------|------|--------|--------|---------
   1 | United States      |   40 |     44 |     42 |      250
   2 | China              |   40 |     27 |     24 |      198
   3 | France             |   16 |     26 |     22 |      122
   4 | Great Britain      |   14 |     22 |     29 |      115
   5 | Australia          |   18 |     19 |     16 |      108
   6 | Japan              |   20 |     12 |     13 |       97
   7 | Italy              |   12 |     13 |     15 |       77
   8 | Netherlands        |   15 |      7 |     12 |       71
   9 | Germany            |   12 |     13 |      8 |       70
  10 | South Korea        |   13 |      9 |     10 |       67
  11 | Canada             |    9 |      7 |     11 |       5

In [ ]:

# Custom QuickSort Implementation
def partition(arr, start, end):
    """Helper for QuickSort: partition array around a pivot (descending order)."""
    pivot = arr[end][4]  # Pivot is weighted score
    pindex = start
    for i in range(start, end):
        if arr[i][4] >= pivot:  # Modified for descending order
            arr[i], arr[pindex] = arr[pindex], arr[i]
            pindex += 1
    arr[pindex], arr[end] = arr[end], arr[pindex]
    return pindex

def quick_sort(arr, start, end):
    """Sort countries by weighted score (descending) using QuickSort."""
    if start < end:
        pindex = partition(arr, start, end)
        quick_sort(arr, start, pindex - 1)
        quick_sort(arr, pindex + 1, end)
    return arr

# Core Logic
def get_top_k(countries, k):
    """Get top K countries using QuickSort."""
    if k <= 0:
        return []
    # Sort in descending order and return first k elements
    sorted_countries = quick_sort(countries.copy(), 0, len(countries) - 1)
    return sorted_countries[:k]

# User Interaction
def display_top(countries):
    print("\nRank | Country            | Gold | Silver | Bronze | Weighted")
    print("-----|--------------------|------|--------|--------|---------")
    for idx, (name, g, s, b, w) in enumerate(countries, 1):
        print(f"{idx:4} | {name[:18]:18} | {g:4} | {s:6} | {b:6} | {w:8}")

def compare_countries(data, name1, name2):
    """Compare two countries by weighted score using linear search."""
    country1 = linear_search(data, name1)
    country2 = linear_search(data, name2)

    if not country1 or not country2:
        print("Error: One or both countries not found!")
        return

    print(f"\nComparison:")
    print(f"{country1[0]}: {country1[4]} pts (G:{country1[1]}, S:{country1[2]}, B:{country1[3]})")
    print(f"{country2[0]}: {country2[4]} pts (G:{country2[1]}, S:{country2[2]}, B:{country2[3]})")

    if country1[4] > country2[4]:
        print(f"{country1[0]} ranks higher!")
    elif country1[4] < country2[4]:
        print(f"{country2[0]} ranks higher!")
    else:
        print("Both countries have equal scores!")

# Main Program
def main():
    data = load_data('olympics2024.csv')

    while True:
        print("\n==== Olympics 2024 Ranking Based on Weighted Score ====")
        print("1. View Top K Countries")
        print("2. Compare Two Countries")
        print("3. Exit")

        choice = input("Choose an option (1-3): ")

        if choice == '1':
            try:
                k = int(input("Enter the number of top countries you want to find: "))
                if 0 < k <= len(data):
                    top_k = get_top_k(data, k)
                    display_top(top_k)
                elif k <= 0:
                    print("Please enter a positive number!")
                else:
                    print("Invalid number of countries!")
            except ValueError:
                print("Please enter a valid number!")

        elif choice == '2':
            name1 = input("Enter first country: ").strip()
            name2 = input("Enter second country: ").strip()
            compare_countries(data, name1, name2)

        elif choice == '3':
            print("Feel free to visit again!")
            break

        else:
            print("Invalid choice!")

if __name__ == "__main__":
    main()


==== Olympics 2024 Ranking Based on Weighted Score ====
1. View Top K Countries
2. Compare Two Countries
3. Exit
Choose an option (1-3): 1
Enter the number of top countries you want to find: 56

Rank | Country            | Gold | Silver | Bronze | Weighted
-----|--------------------|------|--------|--------|---------
   1 | United States      |   40 |     44 |     42 |      250
   2 | China              |   40 |     27 |     24 |      198
   3 | France             |   16 |     26 |     22 |      122
   4 | Great Britain      |   14 |     22 |     29 |      115
   5 | Australia          |   18 |     19 |     16 |      108
   6 | Japan              |   20 |     12 |     13 |       97
   7 | Italy              |   12 |     13 |     15 |       77
   8 | Netherlands        |   15 |      7 |     12 |       71
   9 | Germany            |   12 |     13 |      8 |       70
  10 | South Korea        |   13 |      9 |     10 |       67
  11 | Canada             |    9 |      7 |     11 |       5

In [ ]:

# Custom BubbleSort Implementation
def bubble_sort(arr):
    """Sort countries by weighted score (descending) using BubbleSort."""
    n = len(arr)
    for k in range(1, n):
        flag = 0
        for i in range(0, n - k):
            if arr[i][4] < arr[i + 1][4]:  # Modified for descending order
                arr[i], arr[i + 1] = arr[i + 1], arr[i]  # Swap
                flag = 1
        if flag == 0:  # Early termination if no swaps
            break
    return arr

# Core Logic
def get_top_k(countries, k):
    """Get top K countries using BubbleSort."""
    if k <= 0:
        return []
    sorted_countries = bubble_sort(countries.copy())  # Avoid modifying original
    return sorted_countries[:k]

# User Interaction
def display_top(countries):
    print("\nRank | Country            | Gold | Silver | Bronze | Weighted")
    print("-----|--------------------|------|--------|--------|---------")
    for idx, (name, g, s, b, w) in enumerate(countries, 1):
        print(f"{idx:4} | {name[:18]:18} | {g:4} | {s:6} | {b:6} | {w:8}")

def compare_countries(data, name1, name2):
    """Compare two countries by weighted score using linear search."""
    country1 = linear_search(data, name1)
    country2 = linear_search(data, name2)

    if not country1 or not country2:
        print("Error: One or both countries not found!")
        return

    print(f"\nComparison:")
    print(f"{country1[0]}: {country1[4]} pts (G:{country1[1]}, S:{country1[2]}, B:{country1[3]})")
    print(f"{country2[0]}: {country2[4]} pts (G:{country2[1]}, S:{country2[2]}, B:{country2[3]})")

    if country1[4] > country2[4]:
        print(f"{country1[0]} ranks higher!")
    elif country1[4] < country2[4]:
        print(f"{country2[0]} ranks higher!")
    else:
        print("Both countries have equal scores!")

# Main Program
def main():
    data = load_data('olympics2024.csv')

    while True:
        print("\n==== Olympics 2024 Ranking Based on Weighted Score ====")
        print("1. View Top K Countries")
        print("2. Compare Two Countries")
        print("3. Exit")

        choice = input("Choose an option (1-3): ")

        if choice == '1':
            try:
                k = int(input("Enter the number of top countries you want to find: "))
                if 0 < k <= len(data):
                    top_k = get_top_k(data, k)
                    display_top(top_k)
                elif k <= 0:
                    print("Please enter a positive number!")
                else:
                    print("Invalid number of countries!")
            except ValueError:
                print("Please enter a valid number!")

        elif choice == '2':
            name1 = input("Enter first country: ").strip()
            name2 = input("Enter second country: ").strip()
            compare_countries(data, name1, name2)

        elif choice == '3':
            print("Feel free to visit again!")
            break

        else:
            print("Invalid choice!")

if __name__ == "__main__":
    main()


==== Olympics 2024 Ranking Based on Weighted Score ====
1. View Top K Countries
2. Compare Two Countries
3. Exit
Choose an option (1-3): 1
Enter the number of top countries you want to find: 78

Rank | Country            | Gold | Silver | Bronze | Weighted
-----|--------------------|------|--------|--------|---------
   1 | United States      |   40 |     44 |     42 |      250
   2 | China              |   40 |     27 |     24 |      198
   3 | France             |   16 |     26 |     22 |      122
   4 | Great Britain      |   14 |     22 |     29 |      115
   5 | Australia          |   18 |     19 |     16 |      108
   6 | Japan              |   20 |     12 |     13 |       97
   7 | Italy              |   12 |     13 |     15 |       77
   8 | Netherlands        |   15 |      7 |     12 |       71
   9 | Germany            |   12 |     13 |      8 |       70
  10 | South Korea        |   13 |      9 |     10 |       67
  11 | Canada             |    9 |      7 |     11 |       5

In [ ]:
import time
from copy import deepcopy

# --- TimSort + Binary Search ---
def tim_sort(arr):
    arr.sort(key=lambda x: x[4], reverse=True)
    return arr

def binary_search(arr, target):
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        mid_country = arr[mid][0].lower()
        if mid_country == target.lower():
            return arr[mid]
        elif mid_country < target.lower():
            high = mid - 1
        else:
            low = mid + 1
    return None

# --- QuickSort + Linear Search ---
def quick_sort(arr, start, end):
    if start < end:
        pindex = partition(arr, start, end)
        quick_sort(arr, start, pindex - 1)
        quick_sort(arr, pindex + 1, end)
    return arr

def partition(arr, start, end):
    pivot = arr[end][4]
    pindex = start
    for i in range(start, end):
        if arr[i][4] >= pivot:
            arr[i], arr[pindex] = arr[pindex], arr[i]
            pindex += 1
    arr[pindex], arr[end] = arr[end], arr[pindex]
    return pindex

def linear_search(countries, name):
    for country in countries:
        if country[0].lower() == name.lower():
            return country
    return None

# --- BubbleSort + Linear Search ---
def bubble_sort(arr):
    n = len(arr)
    for k in range(1, n):
        flag = 0
        for i in range(0, n - k):
            if arr[i][4] < arr[i + 1][4]:
                arr[i], arr[i + 1] = arr[i + 1], arr[i]
                flag = 1
        if flag == 0:
            break
    return arr

# --- Benchmarking ---
def benchmark_sorting(data):
    print("\n== Sorting Time Comparison (in seconds) ==")

    # TimSort
    data1 = deepcopy(data)
    start = time.time()
    tim_sort(data1)
    print("Top after TimSort:", data1[:1])
    end = time.time()
    print(f"TimSort: {end - start:.6f}")

    # QuickSort
    data2 = deepcopy(data)
    start = time.time()
    quick_sort(data2, 0, len(data2) - 1)
    print("Top after QuickSort:", data2[:1])
    end = time.time()
    print(f"QuickSort: {end - start:.6f}")

    # BubbleSort
    data3 = deepcopy(data)
    start = time.time()
    bubble_sort(data3)
    print("Top after BubbleSort:", data3[:1])
    end = time.time()
    print(f"BubbleSort: {end - start:.6f}")

# Run everything
data = load_data("olympics2024.csv")
benchmark_sorting(data)


== Sorting Time Comparison (in seconds) ==
Top after TimSort: [('United States', 40, 44, 42, 250)]
TimSort: 0.000048
Top after QuickSort: [('United States', 40, 44, 42, 250)]
QuickSort: 0.000356
Top after BubbleSort: [('United States', 40, 44, 42, 250)]
BubbleSort: 0.000211
